In [2]:
import os

In [3]:
%pwd

'c:\\Users\\nimba\\Coder\\EndToEndDataScienceProject\\research'

In [4]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\nimba\\Coder\\EndToEndDataScienceProject'

In [6]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    rot_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [7]:
from src.EndToEndDataScienceProject.constants import *


In [8]:
from src.EndToEndDataScienceProject.utils.common import read_yaml, create_directories

In [ ]:
class ConfigrationManager:
    def __init__(self, 
                    config_file_path = CONFIG_FILE_PATH,
                    params_file_path = PARAMS_FILE_PATH,
                    schema_file_path = SCHEMA_FILE_PATH):
                    
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        self.schema = read_yaml(schema_file_path)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
    
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            rot_dir = Path(config.root_dir),
            source_URL = config.source_URL,
            local_data_file = Path(config.local_data_file),
            unzip_dir = Path(config.unzip_dir)
        )

        return data_ingestion_config

In [10]:
import os
import urllib.request as request
from src.EndToEndDataScienceProject import logger
import zipfile

In [11]:
## component 1: Data Ingestion

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url=self.config.source_URL,
                filename=self.config.local_data_file
            )
            logger.info(f"{filename} downloaded with following info: \n{headers}")
        else:
            logger.info(f"File already exists")

    def extract_zip_file(self):
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [13]:
try:
    config = ConfigrationManager()
    data_ingestion_config = config.get_data_ingestion_config()

    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-10-01 20:41:51,101 : INFO : EndToEndDataScienceProject : YAML file config\config.yaml loaded successfully.]
[2026-10-01 20:41:51,104 : INFO : EndToEndDataScienceProject : YAML file params.yaml loaded successfully.]
[2026-10-01 20:41:51,109 : INFO : EndToEndDataScienceProject : YAML file schema.yaml loaded successfully.]
[2026-10-01 20:41:51,112 : INFO : EndToEndDataScienceProject : Directory created at: artifacts]
[2026-10-01 20:41:51,115 : INFO : EndToEndDataScienceProject : Directory created at: artifacts/data_ingestion]
[2026-10-01 20:41:51,945 : INFO : EndToEndDataScienceProject : artifacts\data_ingestion\data.zip downloaded with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-O